# Temporal Analysis (NDVI/BSI)

This notebook analyzes **year-on-year (YoY)** and **quarter-on-quarter (QoQ)** NDVI/BSI trends from `data/ghana_parquet` for the 7 main road classes:
`residential`, `service`, `primary`, `secondary`, `tertiary`, `trunk`, `unclassified`.

Focus: temporal signatures that can later be validated against manual road-condition labels.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)

ROOT = Path('data/ghana_parquet')
OUT = Path('data/temporal_analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ['residential', 'service', 'primary', 'secondary', 'tertiary', 'trunk', 'unclassified']
YEARS = [2020, 2021, 2022, 2023]
QMAP = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}

In [3]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path | None = None, repo_name: str = "Sentinel-FYP") -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if p.name == repo_name:
            return p
    raise FileNotFoundError(f"Could not find repo root '{repo_name}' from {start}")

repo_root = find_repo_root()
ROOT = repo_root / "data" / "ghana_parquet"

print("Using parquet root:", ROOT)

frames = []
year_dirs = sorted(ROOT.glob("year=*"))
print("Found year dirs:", [d.name for d in year_dirs])

for ydir in year_dirs:
    files = sorted(ydir.glob("*.parquet"))
    if not files:
        continue
    year = int(ydir.name.split("=", 1)[1])
    df = pd.read_parquet(files[0], columns=["osm_id", "fclass", "quarter", "NDVI", "BSI"])
    df["year"] = year
    frames.append(df)

if not frames:
    raise FileNotFoundError(
        f"No parquet files loaded from {ROOT}. "
        "Check your path and that year=*/ contains .parquet files."
    )

raw = pd.concat(frames, ignore_index=True)
raw["osm_id"] = raw["osm_id"].astype(str)
raw["fclass"] = raw["fclass"].astype(str).str.strip().str.lower()
raw["quarter"] = raw["quarter"].astype(str).str.upper().str.strip()

raw = raw[raw["fclass"].isin(TARGET_CLASSES)]
raw = raw[raw["quarter"].isin(QMAP)]

print("Rows:", len(raw))
print("Classes:", sorted(raw["fclass"].unique().tolist()))
print("Years:", sorted(raw["year"].unique().tolist()))
print("Quarters:", sorted(raw["quarter"].unique().tolist()))


Using parquet root: /Users/miranda/Documents/GitHub/Sentinel-FYP/data/ghana_parquet
Found year dirs: ['year=2020', 'year=2021', 'year=2022', 'year=2023']
Rows: 5265664
Classes: ['primary', 'residential', 'secondary', 'service', 'tertiary', 'trunk', 'unclassified']
Years: [2020, 2021, 2022, 2023]
Quarters: ['Q1', 'Q2', 'Q3', 'Q4']


## 1) Panel Preparation

In [4]:
panel = (
    raw.groupby(['osm_id', 'fclass', 'year', 'quarter'], as_index=False)[['NDVI', 'BSI']]
       .mean(numeric_only=True)
)
panel['quarter_num'] = panel['quarter'].map(QMAP)
panel['t'] = (panel['year'] - panel['year'].min()) * 4 + (panel['quarter_num'] - 1)

panel = panel.sort_values(['osm_id', 'year', 'quarter_num']).reset_index(drop=True)
panel.head()

,osm_id,fclass,year,quarter,NDVI,BSI,quarter_num,t
0,1000044406,residential,2020,Q1,0.201374,0.217902,1,0
1,1000044406,residential,2020,Q2,0.404289,0.111471,2,1
2,1000044406,residential,2020,Q3,0.360927,-0.029027,3,2
3,1000044406,residential,2020,Q4,0.304340,0.159114,4,3
4,1000044406,residential,2021,Q1,0.185572,0.219959,1,4


## 2) Class-Level Yearly Trends (YoY)

In [5]:
class_year = (
    panel.groupby(['fclass', 'year'], as_index=False)[['NDVI', 'BSI']]
         .mean(numeric_only=True)
)

class_year = class_year.sort_values(['fclass', 'year']).reset_index(drop=True)
class_year['ndvi_yoy_change'] = class_year.groupby('fclass')['NDVI'].diff()
class_year['bsi_yoy_change'] = class_year.groupby('fclass')['BSI'].diff()

class_year.to_csv(OUT / 'class_year_trends.csv', index=False)
class_year

,fclass,year,NDVI,BSI,ndvi_yoy_change,bsi_yoy_change
0,primary,2020,0.197832,0.108728,NaN,NaN
1,primary,2021,0.186404,0.109554,-0.011428,0.000826
2,primary,2022,0.201381,0.122022,0.014976,0.012468
3,primary,2023,0.207075,0.115600,0.005695,-0.006422
4,residential,2020,0.259214,0.116862,NaN,NaN
5,residential,2021,0.246974,0.110653,-0.012240,-0.006209
6,residential,2022,0.269072,0.133677,0.022099,0.023024
7,residential,2023,0.267550,0.122166,-0.001522,-0.011511
8,secondary,2020,0.199746,0.105793,NaN,NaN
9,secondary,2021,0.191956,0.106207,-0.007790,0.000414


## 3) Class-Level Quarterly Trends (QoQ)

In [6]:
class_quarter = (
    panel.groupby(['fclass', 'year', 'quarter', 'quarter_num'], as_index=False)[['NDVI', 'BSI']]
         .mean(numeric_only=True)
         .sort_values(['fclass', 'year', 'quarter_num'])
)

class_quarter['ndvi_qoq_change'] = class_quarter.groupby('fclass')['NDVI'].diff()
class_quarter['bsi_qoq_change'] = class_quarter.groupby('fclass')['BSI'].diff()

class_quarter.to_csv(OUT / 'class_quarter_trends.csv', index=False)
class_quarter.head(20)

,fclass,year,quarter,quarter_num,NDVI,BSI,ndvi_qoq_change,bsi_qoq_change
0,primary,2020,Q1,1,0.168282,0.129847,NaN,NaN
1,primary,2020,Q2,2,0.201440,0.104185,0.033158,-0.025662
2,primary,2020,Q3,3,0.220083,0.094956,0.018643,-0.009229
3,primary,2020,Q4,4,0.204709,0.103863,-0.015374,0.008908
4,primary,2021,Q1,1,0.164268,0.115281,-0.040441,0.011417
5,primary,2021,Q2,2,0.184033,0.111347,0.019765,-0.003933
6,primary,2021,Q3,3,0.179341,0.085114,-0.004692,-0.026233
7,primary,2021,Q4,4,0.213854,0.112641,0.034513,0.027527
8,primary,2022,Q1,1,0.164925,0.158229,-0.048930,0.045588
9,primary,2022,Q2,2,0.206043,0.100733,0.041118,-0.057496


## 4) Road-Level Trend Features (for later labeling/validation)

In [7]:
def safe_slope(x, y, min_points=8):
    mask = (~np.isnan(x)) & (~np.isnan(y))
    if mask.sum() < min_points:
        return np.nan
    return float(np.polyfit(x[mask], y[mask], 1)[0])

road_rows = []
for osm_id, g in panel.groupby('osm_id'):
    g = g.sort_values('t')
    x = g['t'].to_numpy(dtype=float)
    ndvi = g['NDVI'].to_numpy(dtype=float)
    bsi = g['BSI'].to_numpy(dtype=float)

    # latest available QoQ and YoY deltas
    g2 = g.copy()
    g2['ndvi_qoq'] = g2['NDVI'].diff()
    g2['bsi_qoq'] = g2['BSI'].diff()

    # YoY by quarter: compare same quarter with previous year where possible
    yoy = g2[['year', 'quarter_num', 'NDVI', 'BSI']].copy()
    yoy_prev = yoy.copy()
    yoy_prev['year'] += 1
    yoy_prev = yoy_prev.rename(columns={'NDVI': 'NDVI_prev', 'BSI': 'BSI_prev'})
    yoy_m = yoy.merge(yoy_prev, on=['year', 'quarter_num'], how='left')
    yoy_m['ndvi_yoy'] = yoy_m['NDVI'] - yoy_m['NDVI_prev']
    yoy_m['bsi_yoy'] = yoy_m['BSI'] - yoy_m['BSI_prev']

    fclass = g['fclass'].mode().iloc[0] if not g['fclass'].mode().empty else g['fclass'].iloc[0]

    road_rows.append({
        'osm_id': osm_id,
        'road_class': fclass,
        'points': len(g),
        'ndvi_trend_slope_per_quarter': safe_slope(x, ndvi),
        'bsi_trend_slope_per_quarter': safe_slope(x, bsi),
        'ndvi_latest_qoq': g2['ndvi_qoq'].dropna().iloc[-1] if g2['ndvi_qoq'].notna().any() else np.nan,
        'bsi_latest_qoq': g2['bsi_qoq'].dropna().iloc[-1] if g2['bsi_qoq'].notna().any() else np.nan,
        'ndvi_latest_yoy': yoy_m['ndvi_yoy'].dropna().iloc[-1] if yoy_m['ndvi_yoy'].notna().any() else np.nan,
        'bsi_latest_yoy': yoy_m['bsi_yoy'].dropna().iloc[-1] if yoy_m['bsi_yoy'].notna().any() else np.nan,
        'ndvi_mean_2023': g.loc[g['year'] == 2023, 'NDVI'].mean(),
        'bsi_mean_2023': g.loc[g['year'] == 2023, 'BSI'].mean(),
    })

road_trends = pd.DataFrame(road_rows)
road_trends.to_csv(OUT / 'road_temporal_features.csv', index=False)
road_trends.head()

,osm_id,road_class,points,ndvi_trend_slope_per_quarter,bsi_trend_slope_per_quarter,ndvi_latest_qoq,bsi_latest_qoq,ndvi_latest_yoy,bsi_latest_yoy,ndvi_mean_2023,bsi_mean_2023
0,1000044406,residential,16,0.003910,0.002077,-0.179925,0.160064,-0.029502,0.045883,0.368845,0.134643
1,1000069142,residential,16,0.002889,0.002485,0.091081,0.054058,0.008089,0.009835,0.386167,0.118303
2,1000268792,service,16,-0.002423,-0.002741,0.058103,-0.064913,-0.001404,0.122369,0.213018,0.148261
3,1000268793,service,16,-0.010592,0.006210,0.189344,-0.173844,0.156687,0.091383,0.453916,0.054750
4,1000268794,service,16,0.000494,-0.001913,0.086073,-0.085474,0.008669,0.020655,0.424399,0.079808


## 5) Indirect Temporal Degradation Flag (Exploratory, No Manual Validation Yet)

In [ ]:
# Exploratory rule: NDVI down + BSI up suggests possible worsening surface/bare exposure over time
tmp = road_trends.copy()

tmp['risk_score'] = 0.0
tmp['risk_score'] += np.where(tmp['ndvi_trend_slope_per_quarter'].notna(), np.maximum(0, -tmp['ndvi_trend_slope_per_quarter'] * 30), 0)
tmp['risk_score'] += np.where(tmp['bsi_trend_slope_per_quarter'].notna(), np.maximum(0,  tmp['bsi_trend_slope_per_quarter'] * 30), 0)
tmp['risk_score'] += np.where(tmp['ndvi_latest_yoy'].notna(), np.maximum(0, -tmp['ndvi_latest_yoy'] * 5), 0)
tmp['risk_score'] += np.where(tmp['bsi_latest_yoy'].notna(), np.maximum(0,  tmp['bsi_latest_yoy'] * 5), 0)

tmp['temporal_risk'] = pd.cut(
    tmp['risk_score'],
    bins=[-np.inf, 0.8, 2.0, np.inf],
    labels=['low', 'med', 'high']
)

tmp.to_csv(OUT / 'road_temporal_risk_labels.csv', index=False)
tmp['temporal_risk'].value_counts(dropna=False)

## 6) Class-wise Summary of Exploratory Temporal Risk

In [ ]:
risk_summary = (
    tmp.groupby(['road_class', 'temporal_risk'], dropna=False)
       .size()
       .reset_index(name='count')
       .sort_values(['road_class', 'temporal_risk'])
)

risk_summary.to_csv(OUT / 'class_temporal_risk_summary.csv', index=False)
risk_summary

## 7) Optional Quick Plot (if matplotlib is installed)

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 5))
    p = class_year.pivot(index='year', columns='fclass', values='BSI')
    p.plot(ax=ax, marker='o')
    ax.set_title('Class-Level BSI by Year')
    ax.set_ylabel('Mean BSI')
    ax.grid(alpha=0.3)
    plt.tight_layout()
except Exception as e:
    print('matplotlib not available or plotting failed:', e)

## Notes
- This notebook intentionally does **not** perform manual-label validation yet.
- Once manual labels are ready, next step is to test correlation/precision-recall between temporal features and manual degradation labels.
- Output CSVs are written to `data/temporal_analysis_outputs/`.